# s05 — The File-System Tool Arsenal

**What this teaches:** what `fstools.New()` actually gives you — the full set of file-system tools that turn a chat model into a useful coding agent: `bash`, `read`, `write`, `grep`, `glob`, `revert` (and `mime`, covered in [s03_mime](../s03_mime/s03_mime.ipynb)).

**Concept anchor:** the agent's *competence* is just the union of its tools. A model with `read`+`write`+`grep`+`glob`+`bash` can navigate, search, edit, and execute — a substantial subset of what a developer does in a terminal. Everything later in this tutorial (skills, sub-agents, MCP) is built on top of this base.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars (see [docs/providers.md](../../docs/providers.md)).
- Write access to the notebook's working directory — the agent will create and delete `hello.txt` there.
- `bash` available on `$PATH` for the `bash` tool (almost certainly already true).

## 1. Imports

Same trio as previous examples — `agentkit`, `stream`, and the `fstools` package whose `New()` returns the bundled tool slice.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
    fstools "github.com/blouargant/omnis/core/tools"
)

## 2. Helper

Panic on error to keep the GoNB kernel alive.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Tour of the tools

`fstools.New()` returns a `[]*tool.FunctionTool` containing — at a minimum:

| Tool | What it does |
|---|---|
| `bash` | Runs a shell command with a safety floor (output truncation, timeout, working-dir guard). |
| `read` | Reads a file (or a line range), returning text suitable for the model. |
| `write` | Creates or overwrites a file; the previous content is snapshotted for `revert`. |
| `grep` | Recursive regex search returning matched lines with `path:line:col` headers. |
| `glob` | Path-glob enumeration (e.g. `**/*.go`) returning a file list. |
| `revert` | Restores the most recent `write` (or `bash` mutation) on a given path. |

The `mime` tool is also bundled — see [s03_mime](../s03_mime/s03_mime.ipynb) for its specifics.

## 4. Construct the model and agent

No `Instruction` here — we let the prompt itself drive which tools fire. The default ADK system prompt plus the tool descriptions are enough for a competent model to plan the steps.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s05_tools",
    Description: "Bash + read + write + grep + glob + revert demo.",
    Model:       llm,
    Tools:       fstools.New(),
})
must(err)

## 5. Build the runner and run

The prompt scripts a small write/read/revert dance. Expect three (or more) tool calls inside the single `RunOnce` — that's the agent loop in full effect.

In [ ]:
r, err := agentkit.Runner("s05", a)
must(err)

prompt := "Use the tools to (1) write hello.txt with 'hi', (2) read it back, (3) revert it. Report each step."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- Distinct tool-call events for `write`, `read`, and `revert` — the loop dispatches each before the model continues.
- After `revert`, `hello.txt` either no longer exists or has reverted to its prior state. See [s06_revert](../s06_revert/s06_revert.ipynb) for a focused look at that tool.
- For a single-tool variant focused purely on arithmetic, contrast with [s02_calc](../s02_calc/s02_calc.ipynb).

## Try it yourself

1. Add a `grep` step to the prompt: "...then grep for 'hi' in the current directory and report matches before reverting."
2. Ask the agent to write 10 files via a single `bash` call and then `glob '*.tmp'` them back — observe how it picks `bash` over multiple `write`s when the prompt nudges it that way.